In [2]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

Python executable: /Volumes/Amin's SSD/PhD_Projects/pd-nsc-scrna/.venv/bin/python
Working directory: /Volumes/Amin's SSD/PhD_Projects/pd-nsc-scrna/notebooks


In [3]:
project_root = Path.cwd().parent

data_path = (
    project_root
    / "data"
    / "raw"
    / "GSE176160_ipsc_ipd_patients_raw.h5ad"
)

print("Project root:", project_root)
print("Data file exists:", data_path.is_file())

Project root: /Volumes/Amin's SSD/PhD_Projects/pd-nsc-scrna
Data file exists: True


In [4]:
import anndata as ad

adata = ad.read_h5ad(data_path, backed="r")

print(adata)

AnnData object with n_obs × n_vars = 30676 × 58735 backed at "/Volumes/Amin's SSD/PhD_Projects/pd-nsc-scrna/data/raw/GSE176160_ipsc_ipd_patients_raw.h5ad"
    obs: 'Demux_state', 'Donor', 'Lane', 'condition', 'age', 'years_of_illness', 'gender'
    layers: None (.X)


In [5]:
adata.obs.head()

,Demux_state,Donor,Lane,condition,age,years_of_illness,gender
index,,,,,,,
AAACCTGCACGTCTCT,SNG,UKERi1E4-R1-003,1,Control,53,NaN,M
AAACCTGCAGTATCTG,SNG,UKERiO3H-R1-005,1,Control,71,NaN,M
AAACCTGTCCAGTATG,SNG,UKERi88H-R1-002,1,Parkinson,63,6.0,F
AAACCTGTCCCAACGG,SNG,UKERiG3G-R1-039,1,Control,69,NaN,F
AAACCTGTCCCGGATG,SNG,UKERi1E4-R1-003,1,Control,53,NaN,M


In [6]:
cell_counts = adata.obs["condition"].value_counts()

donor_counts = (
    adata.obs
    .groupby("condition", observed=True)["Donor"]
    .nunique()
)

print("Cells per condition:")
print(cell_counts)

print("\nDonors per condition:")
print(donor_counts)

Cells per condition:
condition
Parkinson    18967
Control      11709
Name: count, dtype: int64

Donors per condition:
condition
Control      5
Parkinson    7
Name: Donor, dtype: int64


In [7]:
donor_inventory = (
    adata.obs
    .groupby(["Donor", "condition"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values("n_cells", ascending=False)
)

donor_inventory

,Donor,condition,n_cells
3,UKERiC99-R1-007,Parkinson,4591
6,UKERi1E4-R1-003,Control,3449
9,UKERi88H-R1-002,Parkinson,3288
5,UKERiG3G-R1-039,Control,3246
7,UKERiAY6-R1-003,Parkinson,2997
10,UKERiO3H-R1-005,Control,2749
4,UKERiR66-R1-007,Parkinson,2699
0,UKERiJ2C-R1-015,Parkinson,2191
8,UKERiPX7-R1-001,Parkinson,2159
1,UKERi1JF-R1-018,Control,1396


In [8]:
conditions_per_donor = (
    adata.obs
    .groupby("Donor", observed=True)["condition"]
    .nunique()
)

print(conditions_per_donor)

print(
    "\nEach donor has exactly one condition:",
    conditions_per_donor.eq(1).all()
)

Donor
UKERiJ2C-R1-015    1
UKERi1JF-R1-018    1
UKERiM89-R1-005    1
UKERiC99-R1-007    1
UKERiR66-R1-007    1
UKERiG3G-R1-039    1
UKERi1E4-R1-003    1
UKERiAY6-R1-003    1
UKERiPX7-R1-001    1
UKERi88H-R1-002    1
UKERiO3H-R1-005    1
UKERi82A-R1-002    1
Name: condition, dtype: int64

Each donor has exactly one condition: True


In [9]:
missing_metadata = (
    adata.obs
    .isna()
    .sum()
    .to_frame(name="n_missing")
)

missing_metadata["percent_missing"] = (
    missing_metadata["n_missing"] / adata.n_obs * 100
).round(2)

missing_metadata

,n_missing,percent_missing
Demux_state,0,0.00
Donor,0,0.00
Lane,0,0.00
condition,0,0.00
age,0,0.00
years_of_illness,11709,38.17
gender,0,0.00


In [10]:
missing_duration_by_condition = (
    adata.obs
    .assign(
        duration_missing=adata.obs["years_of_illness"].isna()
    )
    .groupby("condition", observed=True)["duration_missing"]
    .agg(["sum", "count"])
    .rename(
        columns={
            "sum": "n_missing",
            "count": "n_cells",
        }
    )
)

missing_duration_by_condition

,n_missing,n_cells
condition,,
Control,11709,11709
Parkinson,0,18967


In [11]:
clinical_columns = [
    "Donor",
    "condition",
    "age",
    "gender",
    "years_of_illness",
]

donor_metadata = (
    adata.obs[clinical_columns]
    .drop_duplicates()
    .sort_values("Donor")
    .reset_index(drop=True)
)

print("Number of donor metadata rows:", len(donor_metadata))

donor_metadata

Number of donor metadata rows: 12


,Donor,condition,age,gender,years_of_illness
0,UKERiJ2C-R1-015,Parkinson,73,M,3.0
1,UKERi1JF-R1-018,Control,42,M,NaN
2,UKERiM89-R1-005,Parkinson,64,M,3.0
3,UKERiC99-R1-007,Parkinson,68,M,7.0
4,UKERiR66-R1-007,Parkinson,54,M,3.0
5,UKERiG3G-R1-039,Control,69,F,NaN
6,UKERi1E4-R1-003,Control,53,M,NaN
7,UKERiAY6-R1-003,Parkinson,37,M,4.0
8,UKERiPX7-R1-001,Parkinson,49,M,1.0
9,UKERi88H-R1-002,Parkinson,63,F,6.0


In [12]:
age_summary = (
    donor_metadata
    .groupby("condition", observed=True)["age"]
    .agg(
        n_donors="size",
        mean_age="mean",
        median_age="median",
        min_age="min",
        max_age="max",
    )
    .round(1)
)

age_summary

,n_donors,mean_age,median_age,min_age,max_age
condition,,,,,
Control,5,60.2,66.0,42,71
Parkinson,7,58.3,63.0,37,73


In [13]:
gender_summary = (
    donor_metadata
    .groupby(["condition", "gender"], observed=True)
    .size()
    .unstack("gender", fill_value=0)
)

gender_summary

gender,F,M
condition,,
Control,2,3
Parkinson,1,6


In [14]:
donor_lane_counts = (
    adata.obs
    .groupby(["Donor", "Lane"], observed=True)
    .size()
    .unstack("Lane", fill_value=0)
)

donor_lane_counts

Lane,1,2,3,4,5,6
Donor,,,,,,
UKERiJ2C-R1-015,318,350,378,390,334,421
UKERi1JF-R1-018,205,221,245,251,246,228
UKERiM89-R1-005,143,172,183,174,193,177
UKERiC99-R1-007,668,708,738,811,844,822
UKERiR66-R1-007,431,414,447,448,506,453
UKERiG3G-R1-039,596,576,567,478,503,526
UKERi1E4-R1-003,528,577,599,603,573,569
UKERiAY6-R1-003,504,476,494,489,506,528
UKERiPX7-R1-001,343,323,361,362,381,389


In [15]:
demultiplexing_summary = (
    adata.obs["Demux_state"]
    .value_counts(dropna=False)
)

demultiplexing_summary

Demux_state
SNG    30676
Name: count, dtype: int64

In [16]:
print("Cell identifiers unique:", adata.obs_names.is_unique)
print("Gene identifiers unique:", adata.var_names.is_unique)

print("\nFirst five cell identifiers:")
print(adata.obs_names[:5].tolist())

print("\nFirst five gene identifiers:")
print(adata.var_names[:5].tolist())

Cell identifiers unique: True
Gene identifiers unique: True

First five cell identifiers:
['AAACCTGCACGTCTCT', 'AAACCTGCAGTATCTG', 'AAACCTGTCCAGTATG', 'AAACCTGTCCCAACGG', 'AAACCTGTCCCGGATG']

First five gene identifiers:
['DDX11L1', 'WASH7P', 'MIR6859-1', 'MIR1302-2HG', 'MIR1302-2']


In [17]:
mitochondrial_mask = adata.var_names.str.startswith("MT-")

mitochondrial_genes = adata.var_names[mitochondrial_mask]

print("Number of mitochondrial genes:", mitochondrial_mask.sum())
print(mitochondrial_genes.tolist())

Number of mitochondrial genes: 37
['MT-TF', 'MT-RNR1', 'MT-TV', 'MT-RNR2', 'MT-TL1', 'MT-ND1', 'MT-TI', 'MT-TQ', 'MT-TM', 'MT-ND2', 'MT-TW', 'MT-TA', 'MT-TN', 'MT-TC', 'MT-TY', 'MT-CO1', 'MT-TS1', 'MT-TD', 'MT-CO2', 'MT-TK', 'MT-ATP8', 'MT-ATP6', 'MT-CO3', 'MT-TG', 'MT-ND3', 'MT-TR', 'MT-ND4L', 'MT-ND4', 'MT-TH', 'MT-TS2', 'MT-TL2', 'MT-ND5', 'MT-ND6', 'MT-TE', 'MT-CYB', 'MT-TT', 'MT-TP']


In [18]:
import numpy as np

count_block = adata.X[:256, :]

print("Block shape:", count_block.shape)
print("Storage dtype:", count_block.dtype)
print("Minimum value:", count_block.min())
print("Maximum value:", count_block.max())

print(
    "All values are finite:",
    np.isfinite(count_block).all(),
)

print(
    "All values are integer-valued:",
    np.equal(count_block, np.floor(count_block)).all(),
)

Block shape: (256, 58735)
Storage dtype: float32
Minimum value: 0.0
Maximum value: 3201.0
All values are finite: True
All values are integer-valued: True


In [19]:
import pandas as pd

block_metrics = pd.DataFrame(
    {
        "total_counts": count_block.sum(axis=1),
        "n_genes_by_counts": np.count_nonzero(count_block, axis=1),
    },
    index=adata.obs_names[:count_block.shape[0]],
)

block_metrics.head()

,total_counts,n_genes_by_counts
index,,
AAACCTGCACGTCTCT,17378.0,3771
AAACCTGCAGTATCTG,13234.0,3823
AAACCTGTCCAGTATG,19998.0,4622
AAACCTGTCCCAACGG,15525.0,4234
AAACCTGTCCCGGATG,17370.0,4117


In [20]:
block_metrics["total_counts_mt"] = (
    count_block[:, mitochondrial_mask].sum(axis=1)
)

block_metrics["pct_counts_mt"] = (
    block_metrics["total_counts_mt"]
    / block_metrics["total_counts"].replace(0, np.nan)
    * 100
)

block_metrics.head()

,total_counts,n_genes_by_counts,total_counts_mt,pct_counts_mt
index,,,,
AAACCTGCACGTCTCT,17378.0,3771,655.0,3.769133
AAACCTGCAGTATCTG,13234.0,3823,210.0,1.586822
AAACCTGTCCAGTATG,19998.0,4622,239.0,1.195120
AAACCTGTCCCAACGG,15525.0,4234,289.0,1.861514
AAACCTGTCCCGGATG,17370.0,4117,453.0,2.607945


In [21]:
qc_chunks = []

for counts_chunk, start, stop in adata.chunked_X(chunk_size=256):
    chunk_metrics = pd.DataFrame(
        {
            "total_counts": counts_chunk.sum(axis=1),
            "n_genes_by_counts": np.count_nonzero(
                counts_chunk, axis=1
            ),
            "total_counts_mt": counts_chunk[
                :, mitochondrial_mask
            ].sum(axis=1),
        },
        index=adata.obs_names[start:stop],
    )

    chunk_metrics["pct_counts_mt"] = (
        chunk_metrics["total_counts_mt"]
        / chunk_metrics["total_counts"].replace(0, np.nan)
        * 100
    )

    qc_chunks.append(chunk_metrics)

qc_metrics = pd.concat(qc_chunks)

print("QC table shape:", qc_metrics.shape)

qc_metrics.head()

QC table shape: (30676, 4)


,total_counts,n_genes_by_counts,total_counts_mt,pct_counts_mt
index,,,,
AAACCTGCACGTCTCT,17378.0,3771,655.0,3.769133
AAACCTGCAGTATCTG,13234.0,3823,210.0,1.586822
AAACCTGTCCAGTATG,19998.0,4622,239.0,1.195120
AAACCTGTCCCAACGG,15525.0,4234,289.0,1.861514
AAACCTGTCCCGGATG,17370.0,4117,453.0,2.607945


In [23]:
qc_columns = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
]

qc_summary = (
    qc_metrics[qc_columns]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .T
)

qc_summary.round(2)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
total_counts,30676.0,13191.47,5848.71,2861.0,3489.75,4941.50,9332.75,12357.00,16133.00,23779.50,31557.25,70048.00
n_genes_by_counts,30676.0,3501.58,964.17,502.0,1338.00,1825.00,2893.00,3473.00,4116.00,5120.00,5973.75,8128.00
pct_counts_mt,30676.0,2.31,1.87,0.0,0.02,0.17,1.56,2.08,2.75,4.48,7.30,86.74


In [24]:
qc_metadata = (
    adata.obs[["Donor", "condition", "Lane"]]
    .join(
        qc_metrics,
        how="left",
        validate="one_to_one",
    )
)

qc_metadata.head()

,Donor,condition,Lane,total_counts,n_genes_by_counts,total_counts_mt,pct_counts_mt
index,,,,,,,
AAACCTGCACGTCTCT,UKERi1E4-R1-003,Control,1,17378.0,3771,655.0,3.769133
AAACCTGCAGTATCTG,UKERiO3H-R1-005,Control,1,13234.0,3823,210.0,1.586822
AAACCTGTCCAGTATG,UKERi88H-R1-002,Parkinson,1,19998.0,4622,239.0,1.195120
AAACCTGTCCCAACGG,UKERiG3G-R1-039,Control,1,15525.0,4234,289.0,1.861514
AAACCTGTCCCGGATG,UKERi1E4-R1-003,Control,1,17370.0,4117,453.0,2.607945


In [26]:
import matplotlib.pyplot as plt

print("Matplotlib is ready.")

ModuleNotFoundError: No module named 'matplotlib'